In [15]:
import pandas as pd
import re, os

In [2]:
ctd_exposure_pm = pd.read_csv("./data/raw/CTD_D052638_ixns_20260609050415.csv")
ctd_exposure_ozone = pd.read_csv("./data/raw/CTD_D010126_exposure_20260609050316.csv")
ctd_exposure_nitric_oxide = pd.read_csv("./data/raw/CTD_D009569_exposure_20260609045231.csv")
ctd_exposure_nitrogen_dioxide = pd.read_csv("./data/raw/CTD_D009585_exposure_20260609050257.csv")

ctd_genes = pd.read_csv("./data/raw/CTD_D052638_genes_20260609050446.csv")
ctd_gene_interactions = pd.read_csv("./data/raw/CTD_D052638_ixns_20260609050415.csv")
ctd_phenotypes = pd.read_csv("./data/raw/CTD_D052638_phenotype_20260609050515.csv")
ctd_diseases = pd.read_csv("./data/raw/CTD_D052638_diseases_20260609050501.csv")

## Analyse des datasets

#### PM

In [3]:
ctd_exposure_pm.head()

,Chemical Name,Chemical ID,CAS RN,Gene Symbol,Gene ID,Interaction,Interaction Actions,References,Organisms
0,Particulate Matter,D052638,NaN,1700001K19RIKL,299330,[Oxygen deficiency co-treated with Blood Gluco...,affects^cotreatment|increases^expression,33729688,Rattus norvegicus
1,Particulate Matter,D052638,NaN,9530082P21RIKL,360487,[Oxygen deficiency co-treated with Blood Gluco...,affects^cotreatment|decreases^expression,33729688,Rattus norvegicus
2,Smoke,D012906,NaN,A1BG,1,Smoke results in decreased expression of A1BG ...,decreases^expression,34520756,Homo sapiens
3,Smoke,D012906,NaN,A1BG-AS1,503538,Smoke results in decreased expression of A1BG-...,decreases^expression,34520756,Homo sapiens
4,Dust,D004391,NaN,A2M,2,[Air Pollutants results in increased abundance...,decreases^expression|increases^abundance,41850516,Homo sapiens


In [4]:
ctd_exposure_pm.info()

<class 'pandas.DataFrame'>
RangeIndex: 66552 entries, 0 to 66551
Data columns (total 9 columns):
 #   Column               Non-Null Count  Dtype  
---  ------               --------------  -----  
 0   Chemical Name        66552 non-null  str    
 1   Chemical ID          66552 non-null  str    
 2   CAS RN               0 non-null      float64
 3   Gene Symbol          66552 non-null  str    
 4   Gene ID              66552 non-null  int64  
 5   Interaction          66552 non-null  str    
 6   Interaction Actions  66552 non-null  str    
 7   References           66552 non-null  str    
 8   Organisms            66518 non-null  str    
dtypes: float64(1), int64(1), str(7)
memory usage: 4.6 MB


In [12]:
ctd_exposure_pm['References'].nunique(dropna=True)

2779

In [14]:
ctd_exposure_pm.columns.tolist()

['Chemical Name',
 'Chemical ID',
 'CAS RN',
 'Gene Symbol',
 'Gene ID',
 'Interaction',
 'Interaction Actions',
 'References',
 'Organisms']

In [19]:
# 1. Chemins des fichiers Exposure Studies
exposure_files = {
    "NO": "./data/raw/CTD_D009569_exposure_20260609045231.csv",
    "NO2": "./data/raw/CTD_D009585_exposure_20260609050257.csv",
    "O3": "./data/raw/CTD_D010126_exposure_20260609050316.csv",
    "PM": "./data/raw/CTD_D052638_exposure_20260609050536.csv",
}

In [20]:
# 2. Fonction de lecture CTD
def read_ctd_csv(path):
    return pd.read_csv(path, comment="#")

In [21]:
# 3. Chargement et concaténation
dfs = []

for pollutant, path in exposure_files.items():
    df = read_ctd_csv(path)
    df["source_pollutant"] = pollutant
    df["source_file"] = os.path.basename(path)
    dfs.append(df)

exposure_df = pd.concat(dfs, ignore_index=True)

In [22]:
# 4. Fonctions utilitaires
def is_non_empty(series):
    return series.notna() & (series.astype(str).str.strip() != "")

def split_ctd_values(series):
    values = []

    for x in series.dropna().astype(str):
        for value in re.split(r"\|", x):
            value = value.strip()
            if value:
                values.append(value)

    return pd.Series(values)

In [23]:
# 5. Nettoyage du PMID
exposure_df["PMID"] = (
    exposure_df["Reference"]
    .astype(str)
    .str.extract(r"(\d+)")[0]
)

# 6. Flags de disponibilité
exposure_df["has_pmid"] = is_non_empty(exposure_df["PMID"])
exposure_df["has_receptor"] = is_non_empty(exposure_df["Receptors"])
exposure_df["has_outcome"] = is_non_empty(exposure_df["Outcome"])

# 7. Filtrage principal
filtered_df = exposure_df[
    exposure_df["has_pmid"]
    & exposure_df["has_receptor"]
    & exposure_df["has_outcome"]
].copy()

In [24]:
# 8. Audit global par polluant
audit_by_pollutant = exposure_df.groupby("source_pollutant").agg(
    total_rows=("Reference", "size"),
    unique_pmids=("PMID", "nunique"),
    rows_with_receptor=("has_receptor", "sum"),
    rows_with_outcome=("has_outcome", "sum"),
)

audit_by_pollutant["filtered_rows"] = (
    filtered_df.groupby("source_pollutant").size()
)

audit_by_pollutant["filtered_unique_pmids"] = (
    filtered_df.groupby("source_pollutant")["PMID"].nunique()
)

print(audit_by_pollutant)

                  total_rows  unique_pmids  rows_with_receptor  \
source_pollutant                                                 
NO                        42            42                  40   
NO2                      274           274                 254   
O3                       166           166                 150   
PM                       821           821                 747   

                  rows_with_outcome  filtered_rows  filtered_unique_pmids  
source_pollutant                                                           
NO                               31             31                     31  
NO2                             229            220                    220  
O3                              137            130                    130  
PM                              628            604                    604  


In [25]:
print("Total lignes exposure :", len(exposure_df))
print("PMID uniques avant filtrage :", exposure_df["PMID"].nunique())
print("Lignes exploitables :", len(filtered_df))
print("PMID uniques exploitables :", filtered_df["PMID"].nunique())

Total lignes exposure : 1303
PMID uniques avant filtrage : 874
Lignes exploitables : 985
PMID uniques exploitables : 644


In [27]:
# 10. Top outcomes
top_outcomes = split_ctd_values(filtered_df["Outcome"]).value_counts().head(20)
print(top_outcomes)

Death                                               107
Cardiovascular Diseases                              79
Asthma                                               63
Respiratory Tract Diseases                           53
Prenatal Exposure Delayed Effects                    42
Premature Birth                                      35
respiratory system process                           33
cognition                                            29
Birth Weight                                         28
DNA methylation on cytosine within a CG sequence     28
regulation of blood pressure                         27
gene expression                                      24
inflammatory response                                23
Respiratory Sounds                                   21
Inflammation                                         21
Lung Neoplasms                                       20
DNA methylation                                      19
Myocardial Infarction                           

In [28]:
# 11. Top populations
top_receptors = split_ctd_values(filtered_df["Receptors"]).value_counts().head(20)
print(top_receptors)

Study subjects                                                  464
Children                                                        201
Pregnant females                                                125
Infants or newborns                                              73
Fetuses                                                          66
Subjects with disease:Asthma                                     66
Workers                                                          49
Subjects with disease:Myocardial Infarction                      24
Controls for disease:Asthma                                      24
Subjects with disease:Diabetes Mellitus                          18
Subjects with disease:Pulmonary Disease, Chronic Obstructive     17
Subjects with gene influence:GSTP1                               13
Subjects with disease:Obesity                                    11
Veterans                                                         11
Subjects with disease:Autism Spectrum Disorder  

In [29]:
# 12. Top pays
top_countries = split_ctd_values(filtered_df["Country"]).value_counts().head(20)
print(top_countries)

United States                341
China                        107
Canada                        61
Sweden                        53
United Kingdom                53
Netherlands                   41
Italy                         40
Spain                         39
Denmark                       39
Taiwan, Province of China     33
Germany                       32
Korea, Republic of            31
France                        30
Japan                         20
Brazil                        18
Belgium                       17
Greece                        17
Norway                        16
Switzerland                   13
Mexico                        12
Name: count, dtype: int64


In [30]:
# 13. Top stressors
top_stressors = split_ctd_values(filtered_df["Stressor Agent"]).value_counts().head(30)
print(top_stressors)

Particulate Matter                  507
Air Pollutants                      447
Nitrogen Dioxide                    280
Ozone                               195
Carbon Monoxide                     134
Sulfur Dioxide                      128
Vehicle Emissions                   104
Tobacco Smoke Pollution             100
Soot                                 79
Nitric Oxide                         50
Nitrogen Oxides                      39
Carbon                               35
Polycyclic Aromatic Hydrocarbons     25
Dust                                 25
Smoke                                21
Lead                                 20
Sulfates                             20
Nitrates                             16
Arsenic                              15
Iron                                 14
Zinc                                 13
Nickel                               12
Potassium                            11
Air Pollutants, Occupational         11
Benzene                               9


In [31]:
# 14. Top medium
top_medium = split_ctd_values(filtered_df["Medium"]).value_counts().head(20)
print(top_medium)

air, ambient              403
air                       318
blood                      42
serum                      34
air, indoor                27
plasma                     23
urine                      17
blood, cord                16
air, outdoor               12
placenta                   12
air, personal              11
air, breath, exhaled        9
blood cell                  7
leukocyte, mononuclear      7
dust, house                 5
breath                      4
cigarette                   3
sperm                       3
leukocyte                   3
blood buffy coat            3
Name: count, dtype: int64


In [32]:
# 15. Top exposure markers
top_markers = split_ctd_values(filtered_df["Exposure Marker"]).value_counts().head(20)
print(top_markers)

Particulate Matter                  742
Nitrogen Dioxide                    501
Ozone                               326
Sulfur Dioxide                      233
Carbon Monoxide                     215
Soot                                118
Nitrogen Oxides                      91
Nitric Oxide                         79
Carbon                               61
Sulfates                             36
Zinc                                 31
Lead                                 28
Air Pollutants                       28
Nickel                               25
Iron                                 25
Nitrates                             25
Copper                               25
Arsenic                              24
Polycyclic Aromatic Hydrocarbons     23
Cotinine                             20
Name: count, dtype: int64


In [33]:
# 16. Recouvrement des PMID entre polluants
pmid_sets = {
    pollutant: set(filtered_df.loc[
        filtered_df["source_pollutant"] == pollutant, "PMID"
    ])
    for pollutant in exposure_files.keys()
}

for p1 in pmid_sets:
    for p2 in pmid_sets:
        if p1 < p2:
            common = len(pmid_sets[p1] & pmid_sets[p2])
            print(f"{p1} ∩ {p2} = {common}")

NO ∩ NO2 = 19
NO ∩ O3 = 8
NO ∩ PM = 27
NO2 ∩ O3 = 91
NO2 ∩ PM = 191
O3 ∩ PM = 117


In [34]:
# 17. Export du corpus filtré
filtered_df.to_csv("./data/processed/ctd_exposure_filtered_pmids.csv", index=False)